# PrideConnect live recommender: train in Colab, publish to the app

This notebook reads de-identified numeric feature snapshots from your app, trains acceptance, usefulness, and Neural Collaborative Filtering (NCF) models, saves the numeric model artifacts to Google Drive, then publishes them to the app. The app runs predictions using those weights; Colab does not need to stay running.

**Privacy:** this notebook receives numeric features, labels, and timestamps only. It does not receive names, emails, profile text, or message contents. Configure `PRIDECONNECT_APP_URL` and `ML_COLAB_SYNC_TOKEN` in Colab Secrets (key icon in the left sidebar). Use a non-production demo database while evaluating.


In [ ]:
from google.colab import drive, userdata
from getpass import getpass
from pathlib import Path
from datetime import datetime, timezone
import json, os, requests
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score

drive.mount('/content/drive')

try:
    APP_URL = userdata.get('PRIDECONNECT_APP_URL').strip().rstrip('/')
except Exception:
    APP_URL = input('Preview app base URL (example: https://your-preview.vercel.app): ').strip().rstrip('/')
try:
    SYNC_TOKEN = userdata.get('ML_COLAB_SYNC_TOKEN').strip()
except Exception:
    SYNC_TOKEN = getpass('ML_COLAB_SYNC_TOKEN (input hidden): ').strip()
if not APP_URL.startswith('https://') or not SYNC_TOKEN:
    raise ValueError('Provide an https app URL and the ML sync token in Colab Secrets.')
HEADERS = {'Authorization': f'Bearer {SYNC_TOKEN}'}
DRIVE_DIR = Path('/content/drive/MyDrive/KQ-PrideConnect/models')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print('Ready. App:', APP_URL)

In [ ]:
response = requests.get(f'{APP_URL}/api/ml/training-data', headers=HEADERS, timeout=60)
response.raise_for_status()
data = response.json()
FEATURES = data['featureNames']
print('Feature count:', len(FEATURES))
print('Acceptance examples:', len(data['acceptance']))
print('Match feedback examples:', len(data['relevance']))
print('NCF resolved interactions:', len(data['ncf']))


In [ ]:
def train_artifact(rows, kind, minimum_rows, minimum_each_class):
    rows = sorted(rows, key=lambda row: row.get('timestamp') or '')
    rows = [row for row in rows if row.get('features') and row.get('label') in (0, 1)]
    labels = np.asarray([int(row['label']) for row in rows], dtype=int)
    positive, negative = int(labels.sum()), int(len(labels) - labels.sum())
    if len(rows) < minimum_rows or min(positive, negative) < minimum_each_class:
        print(f'Skipping {kind}: need at least {minimum_rows} examples and {minimum_each_class} from each class; got n={len(rows)}, positive={positive}, negative={negative}.')
        return None
    X = np.asarray([[float(row['features'].get(name, 0) or 0) for name in FEATURES] for row in rows], dtype=float)
    # Reserve the newest 20% for a time-ordered check before fitting the published model.
    split = max(1, min(len(rows) - 1, int(len(rows) * .8)))
    metrics = {'holdoutCount': 0, 'rocAuc': None, 'brierScore': None, 'accuracy': None}
    if len(set(labels[split:])) == 2:
        check = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=17)
        check.fit(X[:split], labels[:split])
        probs = check.predict_proba(X[split:])[:, 1]
        metrics = {'holdoutCount': int(len(labels[split:])), 'rocAuc': float(roc_auc_score(labels[split:], probs)), 'brierScore': float(brier_score_loss(labels[split:], probs)), 'accuracy': float(accuracy_score(labels[split:], probs >= .5))}
    # Fit final weights on all available labeled rows after recording the temporal check.
    final_model = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=17)
    final_model.fit(X, labels)
    minimum_confident = 12 if kind == 'ACCEPTANCE' else 8
    confidence = float(min(1, max(0, (len(rows) - minimum_confident) / (50 - minimum_confident))))
    version = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    return {'type': kind, 'version': version, 'featureNames': FEATURES, 'weights': {name: float(weight) for name, weight in zip(FEATURES, final_model.coef_[0])}, 'bias': float(final_model.intercept_[0]), 'sampleCount': len(rows), 'positiveCount': positive, 'negativeCount': negative, 'confidence': confidence, 'metrics': metrics}

artifacts = [artifact for artifact in [train_artifact(data['acceptance'], 'ACCEPTANCE', 12, 4), train_artifact(data['relevance'], 'RELEVANCE', 8, 3)] if artifact]
for artifact in artifacts:
    print(artifact['type'], 'n=', artifact['sampleCount'], 'holdout=', artifact['metrics'])


In [ ]:
# Train the Neural Collaborative Filtering model used by the original NCF experiment.
# It learns an 8-number embedding for each pseudonymous member and a small neural
# network that estimates acceptance. Members not seen in training have no embedding;
# those candidates keep using the profile-based adaptive recommender.
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, brier_score_loss, accuracy_score

torch.manual_seed(42)
np.random.seed(42)

ncf_rows = sorted(data['ncf'], key=lambda row: row.get('timestamp') or '')[-6000:]

def ncf_frame(rows):
    return [row for row in rows if row.get('senderKey') and row.get('recipientKey') and row.get('label') in (0, 1)]

ncf_rows = ncf_frame(ncf_rows)
ncf_artifact = None
if len(ncf_rows) < 80 or len({row['senderKey'] for row in ncf_rows} | {row['recipientKey'] for row in ncf_rows}) < 12:
    print(f"NCF not trained: need at least 80 resolved interactions and 12 distinct members; got {len(ncf_rows)} rows.")
else:
    cut = int(len(ncf_rows) * 0.8)
    train_rows, holdout_rows = ncf_rows[:cut], ncf_rows[cut:]
    train_users = sorted({r['senderKey'] for r in train_rows} | {r['recipientKey'] for r in train_rows})
    train_user_index = {key: i for i, key in enumerate(train_users)}
    evaluable = [r for r in holdout_rows if r['senderKey'] in train_user_index and r['recipientKey'] in train_user_index]
    holdout_coverage = len(evaluable) / max(1, len(holdout_rows))

    class NCF(nn.Module):
        def __init__(self, num_users, embedding_dim=8, hidden_dim=16):
            super().__init__()
            self.user_embedding = nn.Embedding(num_users, embedding_dim)
            self.mlp = nn.Sequential(nn.Linear(embedding_dim * 2 + 2, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))
        def forward(self, sender_idx, recipient_idx, request_type_onehot):
            sender = self.user_embedding(sender_idx)
            recipient = self.user_embedding(recipient_idx)
            joined = torch.cat([sender, recipient, request_type_onehot], dim=1)
            return self.mlp(joined).squeeze(1)

    def tensors(rows, indices):
        sender = torch.tensor([indices[r['senderKey']] for r in rows], dtype=torch.long)
        recipient = torch.tensor([indices[r['recipientKey']] for r in rows], dtype=torch.long)
        kind = torch.tensor([[1.0, 0.0] if r['isMentorship'] else [0.0, 1.0] for r in rows], dtype=torch.float32)
        labels = torch.tensor([float(r['label']) for r in rows], dtype=torch.float32)
        return sender, recipient, kind, labels

    def fit_ncf(rows, indices, epochs=200):
        model = NCF(len(indices))
        sender, recipient, kind, labels = tensors(rows, indices)
        positive = max(1.0, float(labels.sum()))
        negative = max(1.0, float(len(labels) - labels.sum()))
        criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(negative / positive))
        optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-2)
        for _ in range(epochs):
            model.train()
            optimizer.zero_grad()
            loss = criterion(model(sender, recipient, kind), labels)
            loss.backward()
            optimizer.step()
        return model

    if len(evaluable) < 20 or len({r['label'] for r in evaluable}) < 2 or holdout_coverage < 0.4:
        print(f"NCF not eligible for evaluation: holdout rows={len(evaluable)}, coverage={holdout_coverage:.1%}; need at least 20 rows and both outcomes.")
    else:
        check_model = fit_ncf(train_rows, train_user_index)
        hold_sender, hold_recipient, hold_kind, hold_labels = tensors(evaluable, train_user_index)
        check_model.eval()
        with torch.no_grad():
            hold_probs = torch.sigmoid(check_model(hold_sender, hold_recipient, hold_kind)).numpy()
        hold_true = hold_labels.numpy().astype(int)
        hold_auc = float(roc_auc_score(hold_true, hold_probs))
        hold_brier = float(brier_score_loss(hold_true, hold_probs))
        hold_accuracy = float(accuracy_score(hold_true, hold_probs >= 0.5))
        print(f"NCF chronological holdout: AUC={hold_auc:.3f}, Brier={hold_brier:.3f}, accuracy={hold_accuracy:.3f}, coverage={hold_coverage:.1%}")

        # Train the final NCF artifact. It is saved and reported even when the
        # holdout check fails, but the app activates it only after it clears the gate.
        all_users = sorted({r['senderKey'] for r in ncf_rows} | {r['recipientKey'] for r in ncf_rows})
        all_user_index = {key: i for i, key in enumerate(all_users)}
        final_model = fit_ncf(ncf_rows, all_user_index)
        state = final_model.state_dict()
        user_embeddings = {key: [float(v) for v in state['user_embedding.weight'][idx].tolist()] for key, idx in all_user_index.items()}
        ncf_passed = hold_auc >= 0.52 and holdout_coverage >= 0.4 and len(evaluable) >= 20
        ncf_confidence = float(min(1.0, len(ncf_rows) / 500.0) * min(1.0, max(0.0, (hold_auc - 0.5) / 0.15))) if ncf_passed else 0.0
        ncf_artifact = {
            'type': 'NCF',
            'active': ncf_passed,
            'version': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'),
            'featureNames': ['senderKey', 'recipientKey', 'requestTypeOneHot'],
            'weights': {
                'userEmbeddings': user_embeddings,
                'layer1': {'weights': state['mlp.0.weight'].tolist(), 'bias': state['mlp.0.bias'].tolist()},
                'layer2': {'weights': state['mlp.2.weight'].tolist(), 'bias': state['mlp.2.bias'].tolist()},
            },
            'bias': 0.0,
            'sampleCount': len(ncf_rows),
            'positiveCount': int(sum(r['label'] == 1 for r in ncf_rows)),
            'negativeCount': int(sum(r['label'] == 0 for r in ncf_rows)),
            'confidence': ncf_confidence,
            'metrics': {'holdoutRocAuc': hold_auc, 'holdoutBrierScore': hold_brier, 'holdoutAccuracy': hold_accuracy, 'holdoutCoverage': float(holdout_coverage), 'holdoutCount': len(evaluable)},
        }
        artifacts.append(ncf_artifact)
        if ncf_passed:
            print('NCF passed holdout checks and will contribute to live matches.')
        else:
            print('NCF evaluation is saved for the admin report, but NCF will not affect live matches until quality and coverage pass.')

if not artifacts:
    raise RuntimeError('No model met its minimum evidence and quality checks. Collect more outcomes/feedback, then run this notebook again.')
for artifact in artifacts:
    print(artifact['type'], 'n=', artifact['sampleCount'], 'metrics=', artifact.get('metrics', {}))


In [ ]:
# Save model weights and evaluation metrics to Drive. Raw training examples are not saved.
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
artifact_path = DRIVE_DIR / f'models-{run_id}.json'
artifact_path.write_text(json.dumps({'createdAt': run_id, 'artifacts': artifacts}, indent=2))
print('Saved model artifacts to:', artifact_path)

# Publish active artifact versions to the app. The app stores these in its database
# and uses them for inference even after this Colab runtime disconnects.
publish = requests.post(f'{APP_URL}/api/ml/artifacts', headers={**HEADERS, 'Content-Type': 'application/json'}, json={'artifacts': artifacts}, timeout=60)
if not publish.ok:
    raise RuntimeError(f'Publish failed ({publish.status_code}): {publish.text[:500]}')
print('Published:', publish.json())